# 14 – Tính lại toàn bộ số liệu từ dự đoán gốc (kiểm toán)

Không train, không trích đặc trưng mới. Notebook đọc **dự đoán gốc** trên Drive và tính lại từ đầu các con số trong bài,
để có thể đánh dấu chúng là "đã tính lại" thay vì chỉ "khớp với file JSON":

1. Bảng I (thống kê dữ liệu, trùng người nói) từ `metadata.csv`.
2. Bảng II từ `pred_*.npy` (CI theo câu và theo người nói).
3. Bảng VII, chênh lệch ghép cặp, hiệu chuẩn trước temperature scaling, từng tỉnh, từ `test_probs_*.npy`.
4. Bảng III (các tập con kiểm soát) với cùng bộ nhận dạng địa danh như notebook 03/10.
5. Khoảng cách lỗi (267/682 km) và kiểm định Mantel, **cho cùng một mô hình CE** (baseline notebook 02).
6. Giới tính.
7. Thử nghiệm kênh thu: CI ghép cặp theo người nói (own − norm, …) và mức tham chiếu của tỉnh "cho phổ" (donor).

- **Runtime:** CPU đủ cho mục 1–6 (khoảng 3–5 phút). Mục 7 nạp đặc trưng đã làm nhiễu (chỉ tập test) và head, khoảng 2 phút.
- **Cần đã chạy:** 01, 02, 10, 12.
- **Kết quả:** `results/audit/recompute_report.json`. Gửi file này cho Claude.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/vimd_dl"
PROVINCE_COL, TEXT_COL = "province_code", "text"
N_BOOT, N_PERM = 2000, 9999
CONDS = ["ltas_own", "ltas_norm", "ltas_swap", "telephone", "noise20"]

In [ ]:
import os, re, json, unicodedata, numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.stats import spearmanr, rankdata, chi2_contingency
from sklearn.metrics import f1_score, top_k_accuracy_score
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
RES_DIR = f"{DRIVE_ROOT}/results"; FIN = f"{RES_DIR}/final"; ROB = f"{RES_DIR}/robustness"; OUT = f"{RES_DIR}/audit"
os.makedirs(OUT, exist_ok=True)
r4 = lambda x: round(float(x), 4)
meta = pd.read_csv(f"{DRIVE_ROOT}/metadata.csv", low_memory=False)
meta[PROVINCE_COL] = meta[PROVINCE_COL].astype(str)
prov = pd.read_csv(f"{DRIVE_ROOT}/provinces.csv"); prov[PROVINCE_COL] = prov[PROVINCE_COL].astype(str)
CLASSES = sorted(meta[PROVINCE_COL].unique()); C = len(CLASSES)
y = meta[PROVINCE_COL].map({c: i for i, c in enumerate(CLASSES)}).values
use = ~meta["exclude"].astype(bool).values
split = meta["split"].values
tr, va, te = [(split == s) & use for s in ("train", "valid", "test")]
pidx = prov.set_index(PROVINCE_COL)
NAMES = [pidx.loc[c, "province_name"] for c in CLASSES]
region_of = np.array([pidx.loc[c, "region"] for c in CLASSES])
yt = y[te]; n_te = len(yt)
REPORT = {}

In [ ]:
GEO = {  # key: tên trong provinces.csv -> (tên tiếng Việt, vĩ độ, kinh độ, [địa danh phụ])
 "CaoBang": ("Cao Bằng", 22.666, 106.258, []), "LangSon": ("Lạng Sơn", 21.853, 106.761, []),
 "QuangNinh": ("Quảng Ninh", 20.951, 107.080, ["Hạ Long", "Cẩm Phả", "Móng Cái"]),
 "HaiPhong": ("Hải Phòng", 20.845, 106.688, []), "ThaiBinh": ("Thái Bình", 20.446, 106.336, []),
 "NamDinh": ("Nam Định", 20.420, 106.168, []), "PhuTho": ("Phú Thọ", 21.322, 105.402, ["Việt Trì"]),
 "ThaiNguyen": ("Thái Nguyên", 21.594, 105.848, []), "YenBai": ("Yên Bái", 21.705, 104.875, []),
 "TuyenQuang": ("Tuyên Quang", 21.823, 105.214, []), "HaGiang": ("Hà Giang", 22.823, 104.984, []),
 "LaoCai": ("Lào Cai", 22.486, 103.971, ["Sa Pa"]), "LaiChau": ("Lai Châu", 22.396, 103.458, []),
 "SonLa": ("Sơn La", 21.326, 103.917, []), "DienBien": ("Điện Biên", 21.386, 103.023, []),
 "HoaBinh": ("Hòa Bình", 20.817, 105.338, []), "HaNoi": ("Hà Nội", 21.028, 105.854, []),
 "HaiDuong": ("Hải Dương", 20.940, 106.333, []), "NinhBinh": ("Ninh Bình", 20.251, 105.975, []),
 "VinhPhuc": ("Vĩnh Phúc", 21.310, 105.597, ["Vĩnh Yên"]), "HungYen": ("Hưng Yên", 20.646, 106.051, []),
 "HaNam": ("Hà Nam", 20.541, 105.914, ["Phủ Lý"]), "BacKan": ("Bắc Kạn", 22.147, 105.835, ["Bắc Cạn"]),
 "BacGiang": ("Bắc Giang", 21.273, 106.195, []), "BacNinh": ("Bắc Ninh", 21.186, 106.076, []),
 "ThanhHoa": ("Thanh Hóa", 19.807, 105.776, []), "NgheAn": ("Nghệ An", 18.679, 105.682, []),
 "HaTinh": ("Hà Tĩnh", 18.343, 105.906, []), "QuangBinh": ("Quảng Bình", 17.468, 106.622, ["Đồng Hới"]),
 "QuangTri": ("Quảng Trị", 16.816, 107.100, ["Đông Hà"]), "ThuaThienHue": ("Thừa Thiên Huế", 16.464, 107.586, ["Huế"]),
 "DaNang": ("Đà Nẵng", 16.054, 108.202, []), "QuangNam": ("Quảng Nam", 15.573, 108.474, ["Tam Kỳ", "Hội An"]),
 "QuangNgai": ("Quảng Ngãi", 15.120, 108.792, []), "BinhDinh": ("Bình Định", 13.776, 109.224, ["Quy Nhơn"]),
 "PhuYen": ("Phú Yên", 13.096, 109.309, ["Tuy Hòa"]), "KhanhHoa": ("Khánh Hòa", 12.239, 109.197, ["Nha Trang", "Cam Ranh"]),
 "NinhThuan": ("Ninh Thuận", 11.565, 108.988, ["Phan Rang"]), "BinhThuan": ("Bình Thuận", 10.928, 108.102, ["Phan Thiết"]),
 "KonTum": ("Kon Tum", 14.350, 108.000, []), "GiaLai": ("Gia Lai", 13.983, 108.000, ["Pleiku", "Plây Cu"]),
 "DakLak": ("Đắk Lắk", 12.667, 108.038, ["Buôn Ma Thuột", "Đắc Lắc"]), "DakNong": ("Đắk Nông", 12.004, 107.690, ["Gia Nghĩa"]),
 "LamDong": ("Lâm Đồng", 11.940, 108.458, ["Đà Lạt"]), "HoChiMinh": ("Hồ Chí Minh", 10.776, 106.701, ["Sài Gòn"]),
 "DongNai": ("Đồng Nai", 10.957, 106.843, ["Biên Hòa"]), "BinhDuong": ("Bình Dương", 10.980, 106.651, ["Thủ Dầu Một"]),
 "BaRiaVungTau": ("Bà Rịa", 10.496, 107.169, ["Vũng Tàu"]), "TayNinh": ("Tây Ninh", 11.310, 106.098, []),
 "BinhPhuoc": ("Bình Phước", 11.535, 106.883, ["Đồng Xoài"]), "LongAn": ("Long An", 10.535, 106.413, []),
 "TienGiang": ("Tiền Giang", 10.360, 106.360, ["Mỹ Tho"]), "BenTre": ("Bến Tre", 10.241, 106.376, []),
 "VinhLong": ("Vĩnh Long", 10.254, 105.972, []), "TraVinh": ("Trà Vinh", 9.935, 106.345, []),
 "CanTho": ("Cần Thơ", 10.045, 105.747, []), "HauGiang": ("Hậu Giang", 9.784, 105.470, ["Vị Thanh"]),
 "SocTrang": ("Sóc Trăng", 9.603, 105.980, []), "BacLieu": ("Bạc Liêu", 9.294, 105.727, []),
 "CaMau": ("Cà Mau", 9.176, 105.150, []), "DongThap": ("Đồng Tháp", 10.460, 105.633, ["Cao Lãnh", "Sa Đéc"]),
 "AnGiang": ("An Giang", 10.386, 105.435, ["Long Xuyên", "Châu Đốc"]), "KienGiang": ("Kiên Giang", 10.012, 105.081, ["Rạch Giá", "Phú Quốc"]),
}


TONE_FIX = {"oà": "òa", "oá": "óa", "oả": "ỏa", "oã": "õa", "oạ": "ọa", "oè": "òe", "oé": "óe", "oẻ": "ỏe",
            "oẽ": "õe", "oẹ": "ọe", "uỳ": "ùy", "uý": "úy", "uỷ": "ủy", "uỹ": "ũy", "uỵ": "ụy"}

def norm(s):
    s = unicodedata.normalize("NFC", str(s).lower())
    for a, b in TONE_FIX.items():
        s = s.replace(a, b)
    return re.sub(r"\s+", " ", s)

place_terms = {k: [norm(GEO[n][0])] + [norm(a) for a in GEO[n][3]] for k, n in enumerate(NAMES)}
all_terms = sorted({t for ts in place_terms.values() for t in ts}, key=len, reverse=True)
PLACE_RE = re.compile(r"(?<!\w)(" + "|".join(map(re.escape, all_terms)) + r")(?!\w)")
own_re = {k: re.compile(r"(?<!\w)(" + "|".join(map(re.escape, ts)) + r")(?!\w)") for k, ts in place_terms.items()}
LAT = np.array([GEO[n][1] for n in NAMES]); LON = np.array([GEO[n][2] for n in NAMES]); p_ = np.pi / 180
DIST = 2 * 6371 * np.arcsin(np.sqrt(np.sin((LAT[None] - LAT[:, None]) * p_ / 2) ** 2 + np.cos(LAT[:, None] * p_)
                                    * np.cos(LAT[None] * p_) * np.sin((LON[None] - LON[:, None]) * p_ / 2) ** 2))
nn_km = np.sort(DIST, 1)[:, 1]
REPORT["geography"] = dict(n_place_terms=len(all_terms), nearest_neighbour_km=dict(median=round(float(np.median(nn_km)), 1), max=round(float(nn_km.max()), 1), min=round(float(nn_km.min()), 1)))
print(REPORT["geography"])

## 1. Bảng I

In [ ]:
spk_all = meta["spk"].astype(str).values
dur = meta["duration"].astype(float).values if "duration" in meta else None
upp = meta.groupby(PROVINCE_COL).size(); spp = meta.groupby(PROVINCE_COL)["spk"].nunique()
sets = {s: set(spk_all[split == s]) for s in ("train", "valid", "test")}
sets_used = {s: set(spk_all[(split == s) & use]) for s in ("train", "valid", "test")}
T1 = dict(n_utt=int(len(meta)), per_split={s: int((split == s).sum()) for s in ("train", "valid", "test")},
          per_split_used={s: int(m.sum()) for s, m in (("train", tr), ("valid", va), ("test", te))},
          hours=round(float(dur.sum() / 3600), 2) if dur is not None else None, mean_sec=round(float(dur.mean()), 2) if dur is not None else None,
          n_speakers=int(len(set(spk_all))), spk_per_province=[int(spp.min()), int(spp.max())],
          utt_per_province=[int(upp.min()), int(upp.max()), float(upp.median())],
          regions={k: int(v) for k, v in pd.Series(region_of).value_counts().items()},
          overlap_raw={f"{a}&{b}": len(sets[a] & sets[b]) for a, b in (("train", "valid"), ("train", "test"), ("valid", "test"))},
          overlap_after_exclude={f"{a}&{b}": len(sets_used[a] & sets_used[b]) for a, b in (("train", "valid"), ("train", "test"), ("valid", "test"))})
spk_te = spk_all[te]; _, spk_id = np.unique(spk_te, return_inverse=True); K = spk_id.max() + 1; cnt = np.bincount(spk_id)
T1["test_speakers"] = dict(n=int(K), share_multi=r4((cnt > 1).mean()), max=int(cnt.max()))
print(json.dumps(T1, indent=1)); REPORT["table1"] = T1

rng = np.random.default_rng(0)
B_SPK = rng.integers(0, K, (N_BOOT, K)); B_UTT = rng.integers(0, n_te, (N_BOOT, n_te)); spk_n = np.bincount(spk_id, minlength=K).astype(float)
def ci_spk(v):
    s = np.bincount(spk_id, weights=v, minlength=K)
    return [r4(x) for x in np.percentile(s[B_SPK].sum(1) / spk_n[B_SPK].sum(1), [2.5, 97.5])]
def ci_utt(v):
    return [r4(x) for x in np.percentile(v[B_UTT].mean(1), [2.5, 97.5])]

def metrics(pred, P=None):
    d = DIST[yt, pred]; r = dict(acc=r4((pred == yt).mean()), macro_f1=r4(f1_score(yt, pred, average="macro")),
                                 region_acc=r4((region_of[pred] == region_of[yt]).mean()), km=round(float(d.mean()), 2),
                                 within100=r4((d <= 100).mean()), acc_ci_utt=ci_utt((pred == yt).astype(float)), acc_ci_spk=ci_spk((pred == yt).astype(float)))
    if P is not None:
        r["top5"] = r4(top_k_accuracy_score(yt, P, k=5, labels=np.arange(C)))
    return r

## 2. Bảng II và 3. Bảng VII

In [ ]:
PRED, PROBS = {}, {}
for n in ["acoustic_ltas", "acoustic_mfcc", "text_tfidf", "text_phobert", "xlsr_best_layer_linear", "xlsr_wsum_linear", "xlsr_wsum_mlp"]:
    f = f"{RES_DIR}/pred_{n}.npy"
    if os.path.exists(f):
        PRED[n] = np.load(f)
for k, f in [("B", "B_xlsr_ls"), ("C", "C_phowhisper_ls"), ("D_ls", "D_fusion_ls"), ("D_geols", "D_fusion_geols"), ("ENS", "FINAL_acc")]:
    PROBS[k] = np.load(f"{FIN}/test_probs_{f}.npy"); PRED[k] = PROBS[k].argmax(1)
REPORT["table2"] = {n: metrics(PRED[n]) for n in PRED if n not in PROBS}
REPORT["table7"] = {k: metrics(PRED[k], PROBS[k]) for k in PROBS}
print(pd.DataFrame(REPORT["table2"]).T.to_string()); print(pd.DataFrame(REPORT["table7"]).T.to_string())
PAIRS = [("D_geols", "D_ls"), ("D_ls", "B"), ("D_ls", "C"), ("ENS", "D_ls"), ("D_geols", "xlsr_wsum_mlp")]
PD = {}
for a, b in PAIRS:
    da = (PRED[a] == yt).astype(float) - (PRED[b] == yt); dk = DIST[yt, PRED[a]] - DIST[yt, PRED[b]]
    PD[f"{a} - {b}"] = dict(acc=r4(da.mean()), acc_ci_spk=ci_spk(da), km=round(float(dk.mean()), 2), km_ci_spk=[round(x, 1) for x in ci_spk(dk)])
REPORT["paired"] = PD; print(json.dumps(PD, indent=1))

def calib(P, yy, bins=15):
    conf = P.max(1); ok = (P.argmax(1) == yy).astype(float); e = np.linspace(0, 1, bins + 1); ece = 0.0
    for lo, hi in zip(e[:-1], e[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(ok[m].mean() - conf[m].mean())
    Y = np.eye(P.shape[1])[yy]
    return dict(ece=r4(ece), brier=r4(((P - Y) ** 2).sum(1).mean()), nll=r4(-np.log(np.clip(P[np.arange(len(yy)), yy], 1e-12, 1)).mean()), mean_conf=r4(conf.mean()))
REPORT["calibration_before_ts"] = {k: calib(PROBS[k], yt) for k in ("D_ls", "D_geols")}
pp = {NAMES[c]: r4((PRED["D_geols"][yt == c] == c).mean()) for c in range(C)}
REPORT["per_province_D_geols"] = dict(sorted(pp.items(), key=lambda x: x[1]))
REPORT["test_utts_per_province"] = dict(mean=round(float(np.bincount(yt).mean()), 1), min=int(np.bincount(yt).min()), max=int(np.bincount(yt).max()))

## 4. Bảng III (tập con kiểm soát)

In [ ]:
texts = np.array([norm(t) for t in meta[TEXT_COL].fillna("")])
mentions_any = np.array([bool(PLACE_RE.search(t)) for t in texts])[te]
mentions_own = np.array([bool(own_re[k].search(t)) for k, t in zip(y, texts)])[te]
w = {k: PRED[k] != yt for k in ["text_tfidf", "text_phobert", "acoustic_ltas", "acoustic_mfcc"]}
SUB = {"all": np.ones(n_te, bool), "no_place_name": ~mentions_any, "own_province_named": mentions_own,
       "text_wrong": w["text_tfidf"], "ltas_wrong": w["acoustic_ltas"], "text_and_ltas_wrong": w["text_tfidf"] & w["acoustic_ltas"],
       "both_text_wrong": w["text_tfidf"] & w["text_phobert"], "both_spectral_wrong": w["acoustic_ltas"] & w["acoustic_mfcc"],
       "all_four_wrong": w["text_tfidf"] & w["text_phobert"] & w["acoustic_ltas"] & w["acoustic_mfcc"]}
T3 = {}
for sn, m in SUB.items():
    T3[sn] = dict(n=int(m.sum()), **{k: r4((PRED[k][m] == yt[m]).mean()) for k in ["xlsr_wsum_mlp", "C", "D_ls", "D_geols", "text_tfidf"]})
REPORT["table3"] = T3; REPORT["place_name_rates_test"] = dict(any=r4(mentions_any.mean()), own=r4(mentions_own.mean()))
print(pd.DataFrame(T3).T.to_string()); print(REPORT["place_name_rates_test"])

## 5. Khoảng cách lỗi và kiểm định Mantel (cùng một mô hình CE: baseline notebook 02)

In [ ]:
iu = np.triu_indices(C, 1)
def sym_rate(pred):
    M = np.zeros((C, C)); np.add.at(M, (yt, pred), 1)
    R = M / np.maximum(M.sum(1, keepdims=True), 1); np.fill_diagonal(R, 0)
    return (R + R.T) / 2
def mantel(pred, seed=0):
    rs = rankdata(sym_rate(pred)[iu]); obs = spearmanr(rs, DIST[iu])[0]; g = np.random.default_rng(seed); null = np.empty(N_PERM)
    for i in range(N_PERM):
        p = g.permutation(C); null[i] = spearmanr(rs, DIST[p][:, p][iu])[0]
    return dict(rho=r4(obs), p=float((1 + (np.abs(null) >= abs(obs)).sum()) / (1 + N_PERM)), null95=[r4(x) for x in np.percentile(null, [2.5, 97.5])])
ERR = {}
for k in ["xlsr_wsum_mlp", "D_ls", "D_geols"]:
    wr = PRED[k] != yt
    ERR[k] = dict(mean_km_wrong=round(float(DIST[yt[wr], PRED[k][wr]].mean()), 1), random_wrong_km=round(float((DIST.sum(1) / (C - 1))[yt[wr]].mean()), 1),
                  mantel=mantel(PRED[k]))
    print(k, ERR[k], flush=True)
REPORT["errors_geography"] = ERR

## 6. Giới tính

In [ ]:
GM = {"1": "male", "1.0": "male", "m": "male", "male": "male", "0": "female", "0.0": "female", "f": "female", "female": "female"}
gender = meta["gender"].astype(str).str.strip().str.lower().map(GM).fillna("unknown").values
tab = pd.crosstab(y[tr], gender[tr])[["female", "male"]]; chi2, pchi, dof, _ = chi2_contingency(tab.values); share = tab["male"] / tab.sum(1)
best = {g: int(pd.Series(y[tr][gender[tr] == g]).value_counts().idxmax()) for g in ("female", "male")}
g_te = gender[te]; gonly = np.array([best.get(g, 0) for g in g_te])
REPORT["gender"] = dict(female_share_test=r4((g_te == "female").mean()), chi2=round(float(chi2), 1), dof=int(dof), p=float(pchi),
                        male_share=[r4(share.min()), r4(share.median()), r4(share.max())], gender_only_acc=r4((gonly == yt).mean()),
                        acc_D_geols={g: r4((PRED["D_geols"][g_te == g] == yt[g_te == g]).mean()) for g in ("male", "female")})
print(REPORT["gender"])

## 7. Thử nghiệm kênh thu: CI ghép cặp và mức tham chiếu donor

In [ ]:
class HeadD(nn.Module):
    def __init__(s, LB, DB, LC, DC):
        super().__init__()
        s.register_buffer("mu_B", torch.zeros(LB, DB)); s.register_buffer("sd_B", torch.ones(LB, DB))
        s.register_buffer("mu_C", torch.zeros(LC, DC)); s.register_buffer("sd_C", torch.ones(LC, DC))
        s.w = nn.ParameterDict({"B": nn.Parameter(torch.zeros(LB)), "C": nn.Parameter(torch.zeros(LC))})
        s.mlp = nn.Sequential(nn.Linear(DB + DC, 512), nn.GELU(), nn.Dropout(0.3), nn.Linear(512, C))
    def forward(s, xb, xc):
        parts = []
        for k, x in (("B", xb), ("C", xc)):
            z = (x.float() - getattr(s, f"mu_{k}")) / getattr(s, f"sd_{k}")
            parts.append((z * s.w[k].softmax(0)[None, :, None]).sum(1))
        return s.mlp(torch.cat(parts, -1))

rng_d = np.random.default_rng(0)                 # cùng cách chọn tỉnh "cho phổ" như notebook 12
DONOR = np.array([rng_d.choice([k for k in range(C) if k != c]) for c in yt])
CP = {("clean", l): PRED[f"D_{l}"] for l in ("ls", "geols")}
avail = [c for c in CONDS if os.path.exists(f"{ROB}/feats_{c}/B.npy")]
for c in avail:
    xb, xc = np.load(f"{ROB}/feats_{c}/B.npy"), np.load(f"{ROB}/feats_{c}/C.npy")
    for l in ("ls", "geols"):
        ps = []
        for s in (0, 1, 2):
            h = HeadD(xb.shape[1], xb.shape[2], xc.shape[1], xc.shape[2]); h.load_state_dict(torch.load(f"{FIN}/head_D_{l}_seed{s}.pt", map_location="cpu")); h.eval()
            with torch.no_grad():
                ps.append(np.concatenate([h(torch.tensor(xb[i:i + 256]), torch.tensor(xc[i:i + 256])).softmax(-1).numpy() for i in range(0, len(xb), 256)]))
        CP[(c, l)] = np.mean(ps, 0).argmax(1)
CH = {}
for l in ("ls", "geols"):
    acc = {c: r4((CP[(c, l)] == yt).mean()) for c in ["clean"] + avail}
    pairs = {}
    for a, b in [("ltas_own", "ltas_norm"), ("ltas_own", "ltas_swap"), ("ltas_swap", "ltas_norm"), ("ltas_own", "clean"), ("ltas_norm", "clean"),
                 ("ltas_swap", "clean"), ("telephone", "clean"), ("noise20", "clean")]:
        if (a, l) in CP and (b, l) in CP:
            d = (CP[(a, l)] == yt).astype(float) - (CP[(b, l)] == yt); pairs[f"{a} - {b}"] = dict(diff=r4(d.mean()), ci_spk=ci_spk(d))
    r = dict(acc=acc, paired=pairs)
    if ("ltas_swap", l) in CP:
        sw = CP[("ltas_swap", l)]; cap = (sw == DONOR).astype(float); cap0 = (CP[("clean", l)] == DONOR).astype(float)
        r["donor"] = dict(rate_swap=r4(cap.mean()), rate_clean=r4(cap0.mean()), increase_ci_spk=ci_spk(cap - cap0),
                          uniform_ref_all_errors=r4((sw != yt).mean() / (C - 1)),
                          uniform_ref_new_errors=r4(cap0.mean() + max(0.0, (sw != yt).mean() - (CP[("clean", l)] != yt).mean()) / (C - 1)))
    CH[l] = r
print(json.dumps(CH, indent=1)); REPORT["channel"] = CH
json.dump(REPORT, open(f"{OUT}/recompute_report.json", "w"), indent=1, ensure_ascii=False, default=float)
print(f"Đã lưu {OUT}/recompute_report.json – gửi file này cho Claude.")